In [14]:
"""
Parse ANADDB output file
Currently able to parse: EO tensor, Raman tensor, polarities.
"""
from itertools import tee
import numpy as np
import re
import pandas as pd
import matplotlib.pyplot as plt

### Reading the file ###
class Anaddb:
    def __init__(self, filename, natom=10):
        with open(filename, 'r') as file:
            self.data = file.readlines()
        self.natom = natom
        
    def ionic_EO(self):
        ionic_EO_tensor = np.zeros((6,3))
        index = self.data.index(" Total EO tensor (pm/V) in Voigt notations\n")+1
        
        for i in range(6):
            ionic_EO_tensor[i] = np.array(self.data[index+i].split(), dtype=float)
            
        return ionic_EO_tensor
    
    def get_EO_tensor(self):
        EO_tensors = pd.DataFrame(columns=["Mode number", "Frequency", "EO tensor"])
        index = self.data.index(" Output of the EO tensor (pm/V) in Voigt notations\n")+6
        
        for mode_number in range(4, (self.natom*3) + 1):
            ### space + digit + . + digit?
            frequency = float(re.findall("\s\d+.\d+", self.data[index])[0])
            
            EO = np.zeros((6, 3))
            for i in range(6):
                EO[i] = np.array(self.data[index+i+1].split(), dtype = float)
                
            EO_tensors.loc[len(EO_tensors.index)] = [mode_number, frequency, EO]
            index +=8
        return EO_tensors
            

    def get_Raman(self, NAC=False):
        Raman_tensors = pd.DataFrame(columns=["Mode", "Freq", "Raman susceptibility"])

        if NAC:
            index = self.data.index(" Raman susceptibility of zone-center phonons, with non-analyticity in the\n")+23
        else:
            index = self.data.index("  Raman susceptibilities of transverse zone-center phonon modes\n")+22

        for mode in np.arange(4, self.natom*3+1):
            freq = float(re.findall("\s\d+.\d+", self.data[index])[0])

            Raman = np.zeros((3,3))
            for i in range(3):
                Raman[i] = np.array(self.data[index+i+1][1:].split(), dtype=float)

            Raman_tensors.loc[len(Raman_tensors.index)] = [mode, freq, Raman]
            index += 6

        return Raman_tensors

    def get_BEC(self):
        bec = pd.DataFrame(columns=["atom", "BEC"])
        index = self.data.index(" Effective charge tensors after \n") + 4
        
        for atom in range(1, self.natom+1):
            bec_tensor = np.zeros((3, 3))
            
            for i in range(3):
                bec_tensor[i] = np.array(self.data[index+i][23:].split(), dtype=float)
            bec.loc[len(bec.index)] = [atom, bec_tensor]
            index += 3
            
        return bec        
            
        
    
    def get_polarity(self):
        polarity = pd.DataFrame(columns=["Mode", "px", "py", "pz", "|p|"])
        index = self.data.index(" Mode effective charges \n")+5
        
        for mode in np.arange(4, self.natom*3+1):
            polarity.loc[len(polarity.index)] = self.data[index][1:].split()
            index += 1

        return polarity
    
    
    def get_disp_mag(self):
        displacement = pd.DataFrame(columns=["Mode", "displacement set"])
        index = self.data.index(" Treat the first list of vectors \n")+93
        
        for mode in range(4, self.natom*3 + 1):
            disp = np.zeros((10, 3))
            
            for i in range(10):
                disp[i] = np.array(self.data[index+2*i][5:].split(), dtype=float) 
            
            displacement.loc[len(displacement.index)] = [mode, disp]
            index += 21
            
        return displacement
    
    


    

In [47]:
"""
Post process after the extract, more human friendly print.

Be careful when using the code.
"""
if __name__ == "__main__":
    LNO = Anaddb("tnlo_4.abo")
    print(LNO.ionic_EO()*(-1))
    
def BORN(i):
    """
    The value of Born Effective Charge
    Args:
        i (integer): the index of atoms
    """
    bec = LNO.get_BEC().loc[i-1, 'BEC']
    return bec

def disp(i, j):
    """
    The magnitude of the phonon eigen displacement. 
    Args:
        i (integer): the index of modes
        j (integer): the index of atoms
    """
    disp_mag = LNO.get_disp_mag().loc[i-4, 'displacement set'][j-1:j][:].transpose()
    return disp_mag

def mode_polarity(i):
    """
    The magnitude of the mode polarity
    Args:
        i (integer): the index of modes
    """    
    mod_pol = 0
    for n in range(1, LNO.natom+1):
        mod_pol += BORN(n)@disp(i,n)
        return mod_pol.transpose() 

[[-0.         -4.78801885 10.07651212]
 [-0.          4.78801885 10.07651212]
 [ 0.         -0.         27.6903241 ]
 [ 0.         15.36850141 -0.        ]
 [15.36850141  0.          0.        ]
 [-4.78801885 -0.          0.        ]]


In [59]:
BORN(5)

array([[-1.622916 ,  0.2943624,  0.1683365],
       [ 0.2217712, -4.058521 , -1.807938 ],
       [ 0.1328344, -1.853342 , -2.675733 ]])

In [60]:
mode_list = mode_polarity(4)
for i in range(5,31):        
    mode_list = np.vstack((mode_list, mode_polarity(i)))
np.savetxt(sys.stdout, np.abs(mode_list))


6.682790357900598565e-05 1.403890640090269502e-03 8.968636730261639330e-21
1.403890640090269502e-03 6.682790357900598565e-05 3.416822414643588252e-20
0.000000000000000000e+00 3.962932886464831245e-19 4.124306741321759513e-03
9.424434656170202571e-04 1.778817824171482175e-03 1.388955490955820554e-20
1.778817824171482175e-03 9.424434656170202571e-04 4.865284191456060036e-20
0.000000000000000000e+00 1.243111592290163810e-19 1.293732108814769823e-03
1.833143325441419767e-04 1.212321264833849006e-03 1.083302752858127905e-20
1.212321264833849006e-03 1.833143325441419767e-04 2.885034804052775603e-20
0.000000000000000000e+00 5.817918270940607989e-19 6.054828641497440184e-03
0.000000000000000000e+00 3.730835542406496219e-19 3.882758204375279763e-03
3.075723130059464325e-03 2.917577478983341896e-03 9.084570599147700571e-20
2.917577478983341896e-03 3.075723130059464325e-03 5.568412786459499461e-20
0.000000000000000000e+00 7.074575651244107658e-20 7.362658133846190431e-04
6.297786342624662245e-04 

In [53]:
disp_mag = disp(4, 1).transpose()
for n in range(2, LNO.natom+1):
    disp_mag = np.vstack((disp_mag, disp(4,n).transpose()))
np.savetxt(sys.stdout, disp_mag)


1.837187520000000090e-04 -1.150073269999999943e-03 0.000000000000000000e+00
1.840575700000000024e-04 1.150019099999999989e-03 0.000000000000000000e+00
5.075751469999999731e-04 7.257532239999999607e-04 0.000000000000000000e+00
5.073613089999999678e-04 -7.259027309999999556e-04 0.000000000000000000e+00
-7.453438759999999512e-04 1.182791209999999905e-03 -1.492492310000000056e-03
-1.170029379999999947e-03 1.006146830000000016e-03 5.639595010000000164e-05
-1.110665150000000097e-03 1.462257459999999907e-03 1.436096360000000067e-03
-7.457125039999999835e-04 -1.182416410000000032e-03 1.492257560000000019e-03
-1.110951430000000012e-03 -1.461990270000000040e-03 -1.436359749999999939e-03
-1.170450030000000060e-03 -1.005897150000000006e-03 -5.589781009999999763e-05


In [61]:
disp_mag = disp(4, 1).transpose()
for j in range(4, 31):
    for n in range(1, LNO.natom+1):
        disp_mag = np.vstack((disp_mag, disp(j,n).transpose()))
np.savetxt(sys.stdout, disp_mag)

1.837187520000000090e-04 -1.150073269999999943e-03 0.000000000000000000e+00
1.837187520000000090e-04 -1.150073269999999943e-03 0.000000000000000000e+00
1.840575700000000024e-04 1.150019099999999989e-03 0.000000000000000000e+00
5.075751469999999731e-04 7.257532239999999607e-04 0.000000000000000000e+00
5.073613089999999678e-04 -7.259027309999999556e-04 0.000000000000000000e+00
-7.453438759999999512e-04 1.182791209999999905e-03 -1.492492310000000056e-03
-1.170029379999999947e-03 1.006146830000000016e-03 5.639595010000000164e-05
-1.110665150000000097e-03 1.462257459999999907e-03 1.436096360000000067e-03
-7.457125039999999835e-04 -1.182416410000000032e-03 1.492257560000000019e-03
-1.110951430000000012e-03 -1.461990270000000040e-03 -1.436359749999999939e-03
-1.170450030000000060e-03 -1.005897150000000006e-03 -5.589781009999999763e-05
1.150073269999999943e-03 1.837187520000000090e-04 0.000000000000000000e+00
-1.150019099999999989e-03 1.840575700000000024e-04 0.000000000000000000e+00
-7.257532

In [46]:
mode_polarity(30)

array([[ 0.00000000e+00],
       [-2.53619287e-21],
       [ 2.63946871e-05]])

In [20]:
polarity = LNO.get_polarity()
b = polarity.loc[:, "|p|"]
polarity

,Mode,px,py,pz,|p|
0,4,4.629780,-0.000000,-0.000000,4.629780
1,5,0.000000,4.629780,-0.000000,4.629780
2,6,0.000000,0.000000,-0.000000,0.000000
3,7,-1.768584,-0.000000,-0.000000,1.768584
4,8,-0.000000,-1.768584,-0.000000,1.768584
5,9,-0.000000,-0.000000,6.900053,6.900053
6,10,-3.591195,-0.000000,-0.000000,3.591195
7,11,-0.000000,-3.591195,-0.000000,3.591195
8,12,-0.000000,-0.000000,0.239492,0.239492
9,13,-0.000000,0.000000,-0.000000,0.000000


In [80]:
raman = LNO.get_Raman()
c = raman.loc[:, "Raman susceptibility"]

max_comp = []
for i in range(27):
    max_comp.append(np.max(np.max(c[i:i+1])))
np.savetxt(sys.stdout, max_comp)


4.836354999999999647e-03
4.836354999999999647e-03
-0.000000000000000000e+00
1.688714999999999921e-03
1.688714999999999921e-03
1.981410499999999852e-02
1.394655999999999977e-03
1.394655999999999977e-03
-0.000000000000000000e+00
-0.000000000000000000e+00
0.000000000000000000e+00
5.378319000000000329e-03
6.788879999999999958e-04
2.207844000000000039e-03
2.207844000000000039e-03
2.236501000000000114e-03
2.236501000000000114e-03
-0.000000000000000000e+00
4.548634000000000184e-03
4.548634000000000184e-03
-0.000000000000000000e+00
4.610420999999999964e-03
4.610420999999999964e-03
3.186334899999999914e-02
2.128433999999999795e-03
2.128433999999999795e-03
0.000000000000000000e+00


In [82]:
frequency = LNO.get_EO_tensor()
d = frequency.loc[:, "Frequency"]
d

0     152.04
1     152.04
2     215.46
3     215.75
4     215.75
5     239.43
6     261.74
7     261.74
8     284.69
9     296.66
10    335.15
11    335.15
12    358.52
13    377.83
14    377.83
15    389.86
16    389.86
17    414.82
18    433.55
19    433.55
20    455.93
21    581.74
22    581.74
23    608.90
24    671.58
25    671.58
26    892.15
Name: Frequency, dtype: float64

In [62]:
for i in range(10,31):
    for j in range(10):
        print(i)

10
10
10
10
10
10
10
10
10
10
11
11
11
11
11
11
11
11
11
11
12
12
12
12
12
12
12
12
12
12
13
13
13
13
13
13
13
13
13
13
14
14
14
14
14
14
14
14
14
14
15
15
15
15
15
15
15
15
15
15
16
16
16
16
16
16
16
16
16
16
17
17
17
17
17
17
17
17
17
17
18
18
18
18
18
18
18
18
18
18
19
19
19
19
19
19
19
19
19
19
20
20
20
20
20
20
20
20
20
20
21
21
21
21
21
21
21
21
21
21
22
22
22
22
22
22
22
22
22
22
23
23
23
23
23
23
23
23
23
23
24
24
24
24
24
24
24
24
24
24
25
25
25
25
25
25
25
25
25
25
26
26
26
26
26
26
26
26
26
26
27
27
27
27
27
27
27
27
27
27
28
28
28
28
28
28
28
28
28
28
29
29
29
29
29
29
29
29
29
29
30
30
30
30
30
30
30
30
30
30
